# பாடம் 03 - முகாமைத்துவ வடிவமைப்பு முறைகள்

இந்த பாடத்தில், திறமையான AI முகாமையாளர்களைப் படைக்க மூன்று அடிப்படையான வடிவமைப்பு முறைகளை ஆராய்கிறோம்:

1. **தெளிவான முகாமையாளர் அத்தியாயங்கள்** — முகாமையாளர்களின் நடத்தை வழிநடத்தும் துல்லியமான, பங்கு வரையறுக்கும் உத்திரவுகளை உருவாக்குதல்  
2. **Pydantic மாதிரிகளுடன் அமைக்கபடத்தப்பட்ட வெளியீடு** — முகாமையாளர் கணிக்கபடும், சரிபார்க்கபடும் தரவை வழங்குவததை உறுதி செய்தல்  
3. **தனித்து பொறுப்பு முகாமையாளர்** — ஒவ்வொரு முகாமையாளர் ஒரே காரியத்தை சிறப்பாக செய்ய வடிவமைத்தல்  

ஒவ்வொரு முறையைவும் **பயண இலக்கு பரிந்துரையாளர்** சூழலுக்கு பயன்படுத்தி, இலக்குகளை பரிந்துரைக்கும், கிடைக்கும் நிலையை சரிபார்க்கும் மற்றும் தொடர்புடைய விவரங்களை கையாளும் ஒரு அமைப்பை படிப்படியாக உருவாக்குவோம்.


## அமைப்பு


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## போட்டர்ன் 1: தெளிவான முகவர் வழிமுறைகள்

பெரும்பாலும் மிக முக்கியமான போட்டர்ன் மிகவும் எளிமையானது: உங்கள் முகவருக்கான தெளிவான, விரிவான வழிமுறைகளை எழுதுதல்.

நல்ல வழிமுறைகள் விவரிக்கின்றன:
- **யார்** முகவர் (பாத்திரம் மற்றும் குரல்)
- **என்ன** செய்ய வேண்டும் (படி-படி பொறுப்புகள்)
- **எப்படி** நடந்து கொள்ள வேண்டும் (கட்டுப்பாடுகள் மற்றும் பாணி)

கீழே, நாம் ஒவ்வொரு பதிலும் உருவாக்கும் தெளிவான வழிமுறைகளுடன் ஒரு பயண கவுன்சிலியர் முகவரை உருவாக்குகிறோம்.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## மாதிரிக் கோர் 2: பைடாண்டிக் மாதிரிகளுடன் அமைப்பு கொண்ட வெளியீடு

வகைமுறை எழுத்து உரையாடலுக்கு பயன்படுகிறது, ஆனால் கீழ்நிலை அமைப்புகள் அமைந்த தரவுகளை வேண்டும்.
**பைடாண்டிக் மாதிரிகள்** மற்றும் **கருவி செயல்பாடு** என்ற இரண்டையும் சேர்த்து, நாம்:

- முகவரின் வெளியீட்டிற்கான துல்லியமான பாணியை வரையறுக்கலாம்
- பதில்களை தானாகச் சான்று பார்க்கலாம்
- முகவரின் முடிவுகளை செயலி தர.logicக்கு நம்பகமாக இணைக்கலாம்

நாம் மேலும் ஒரு கருவியை அறிமுகப்படுத்துகிறோம், அது இலக்கிடம் விவரங்களை வழங்குகிறது, எனவே முகவர் தன்னுடைய பரிந்துரைகளை உண்மையான தரவுகளில் அடிப்படையாகக் கொள்கிறது.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: Single Responsibility Agents

சிக்கலான பணிகள் பல கவனம் செலுத்தப்பட்ட முகவர்களுடன் பகிரப்பட்டு ஒவ்வொருவரும் தனித்தனி பொறுப்புடன் செயல்படுவதால் பயனடைவார்கள்:

- இடங்கள் மற்றும் கிடைக்கைகள் பற்றிய அறிவு கொண்ட **Destination Expert**
- பறப்புகள், ஹோட்டல்கள் மற்றும் பயண திட்டங்களைக் கையாளும் **Logistics Planner**

இது மென்பொருள் பொறியியல் கொள்கையான *பதவீகம் பிரிப்பதன்* கோட்பாட்டுடன் ஒத்துப்போகிறது — ஒவ்வொரு முகவரும் சோதனை செய்ய, பராமரிக்க மற்றும் தனித்து மேம்படுத்த எளிதாக இருக்கும்.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## சுருக்கம்

இந்த பாடத்தில் நாம் ஒரு பயண பரிந்துரையாளர் சூழலில் மூன்று முகவர் வடிவமைப்பு முன்மொழிவுகளை பயன்படுத்தினோம்:

| முன்மொழிவு | முக்கிய கருத்து | பயன் |
|---|---|---|
| **தெளிவான hướngிகள்** | முன்னதாகவே நபர், பொறுப்புகள் மற்றும் கட்டுப்பாடுகளை வரையறுக்கவும் | ஒரே மாதிரியான, பிராண்டைத் தக்க வைத்துக் கொண்டு நடக்கும் முகவர் நடையை உறுதி செய்தல் |
| **நிர்மாணிக்கப்பட்ட வெளியீடு** | பதிலாக பைடான்டிக் மாதிரிகளை பயன்படுத்தவும் | சரிபார்க்கப்பட்ட, இயந்திரம் வாசிக்கக்கூடிய முடிவுகள் |
| **ஒற்றை பொறுப்பு** | ஒவ்வொரு முகவருக்கும் ஒரு கவனமான பணியை வழங்கவும் | சோதனை செய்ய, பராமரிக்க, மற்றும் ஒருங்கிணைக்க எளிது |

இந்த முன்மொழிவுகள் இயங்கிக் கொள்கின்றன — நீங்கள் தெளிவான hướngிகள் மற்றும் நிர்மாணிக்கப்பட்ட வெளியீட்டை ஒரே-பொறுப்பு முகவரில் இணைத்து வலுவான, உற்பத்தி-தயார் சிஸ்டம்களை உருவாக்கலாம்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**விமர்சனம்**:  
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சிக்கிறோம், ஆனால் தானாக செய்யப்பட்ட மொழிபெயர்ப்புகளில் தவறுகள் அல்லது அச்சமைகள் இருக்க வாய்ப்பு உள்ளது என்பதை தயவுசெய்து கவனத்தில் கொள்ளவும். தாய்மொழியில் உள்ள அசல் ஆவணம் அதிகாரப்பூர்வ ஆதாரமாகக் கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு தொழில்முறை மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பின் பயன்பாட்டால் ஏற்படும் எந்த புரிதற்கேடு அல்லது தவறான பொருள் விளக்கம் பற்றியும் நாங்கள் பொறுப்பில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
